In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, sampler
import torchvision.datasets as dset
import torchvision.transforms as T
import torch.nn.functional as F

# הגדרת המכשיר (GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# 1. טעינת הנתונים (CIFAR-10)
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

NUM_TRAIN = 49000
cifar10_train = dset.CIFAR10('./datasets', train=True, download=True, transform=transform)
loader_train = DataLoader(cifar10_train, batch_size=64, sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)))

cifar10_val = dset.CIFAR10('./datasets', train=True, download=True, transform=transform)
loader_val = DataLoader(cifar10_val, batch_size=64, sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN, 50000)))

# פונקציית עזר להשטחה
class Flatten(nn.Module):
    def forward(self, x):
        return x.view(x.size(0), -1)

# 2. בניית המודל (Part V)
model = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),
    nn.Conv2d(64, 128, kernel_size=3, padding=1),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),
    Flatten(),
    nn.Linear(128 * 4 * 4, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, 10)
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 3. פונקציית אימון
def train_model(model, optimizer, epochs=10):
    for e in range(epochs):
        model.train()
        for t, (x, y) in enumerate(loader_train):
            x, y = x.to(device), y.to(device)
            scores = model(x)
            loss = F.cross_entropy(scores, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # בדיקת דיוק על ה-Validation בסוף כל Epoch
        model.eval()
        num_correct, num_samples = 0, 0
        with torch.no_grad():
            for x, y in loader_val:
                x, y = x.to(device), y.to(device)
                scores = model(x)
                _, preds = scores.max(1)
                num_correct += (preds == y).sum()
                num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print(f'Epoch {e+1}, Loss: {loss.item():.4f}, Val Accuracy: {100 * acc:.2f}%')

# התחלת האימון
train_model(model, optimizer)

Using device: cuda


100%|██████████| 170M/170M [00:03<00:00, 48.7MB/s]


Epoch 1, Loss: 1.0474, Val Accuracy: 62.30%
Epoch 2, Loss: 0.7523, Val Accuracy: 69.20%
Epoch 3, Loss: 0.7591, Val Accuracy: 76.50%
Epoch 4, Loss: 0.6038, Val Accuracy: 75.00%
Epoch 5, Loss: 0.5265, Val Accuracy: 78.70%
Epoch 6, Loss: 0.3625, Val Accuracy: 78.40%
Epoch 7, Loss: 0.6317, Val Accuracy: 80.00%
Epoch 8, Loss: 0.6467, Val Accuracy: 78.60%
Epoch 9, Loss: 0.2633, Val Accuracy: 80.00%
Epoch 10, Loss: 0.2238, Val Accuracy: 78.60%
